# Assignment 9: Operating-System Randomness

This notebook checks 1 MiB or more from Linux `/dev/urandom`, OpenSSL `rand`, and the Windows .NET cryptographic random-number API. First run the platform capture scripts described in [README.md](README.md), then run every cell. Binary samples are generated locally and are not committed.

The checks are descriptive statistical tests, not cryptographic proofs. They can identify some distribution defects, but cannot demonstrate unpredictability or guarantee secure generation.

In [1]:
from collections import Counter
from math import erfc, sqrt
from pathlib import Path

MIN_BYTES = 1024 * 1024
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "randomness_assessment" / "generated"
if not DATA_DIR.is_dir():
    DATA_DIR = NOTEBOOK_DIR / "generated"

SAMPLES = {
    "Linux /dev/urandom": DATA_DIR / "linux_urandom.bin",
    "OpenSSL rand": DATA_DIR / "openssl_rand.bin",
    "Windows .NET RNG": DATA_DIR / "windows_dotnet_rng.bin",
}

missing = [str(path) for path in SAMPLES.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Generate all samples before running the analysis. Missing: " + ", ".join(missing)
    )
print("All three sample files were found.")

All three sample files were found.


## Test definitions

- **Byte-frequency chi-square:** compares the 256 observed byte counts with equal expected counts. For 255 degrees of freedom, the 5% critical value is about 293.25.
- **Monobit:** checks whether the fraction of one-bits differs from one half; the reported p-value uses the normal approximation.
- **Runs:** checks the number of adjacent bit transitions against the expected count, with the usual frequency precondition `0.01 < proportion_of_ones < 0.99`.
- **Patterns and dependence:** counts duplicate 16-byte blocks and repeated-byte runs and calculates byte correlations at lags 1, 2, 4, and 16. These are sample diagnostics, not formal proofs of randomness.

In [2]:
def analyze(data: bytes) -> dict:
    if len(data) < MIN_BYTES:
        raise ValueError(f"Sample has {len(data)} bytes; need at least {MIN_BYTES}.")

    size = len(data)
    byte_counts = Counter(data)
    expected = size / 256
    byte_chi_square = sum((byte_counts[value] - expected) ** 2 / expected for value in range(256))

    one_bits = sum(value.bit_count() for value in data)
    total_bits = size * 8
    bit_proportion = one_bits / total_bits
    monobit_z = (one_bits - total_bits / 2) / sqrt(total_bits / 4)
    monobit_p = erfc(abs(monobit_z) / sqrt(2))

    transitions = 0
    previous_lsb = None
    for value in data:
        transitions += ((value ^ (value >> 1)) & 0x7F).bit_count()
        if previous_lsb is not None:
            transitions += previous_lsb ^ (value >> 7)
        previous_lsb = value & 1
    runs = transitions + 1
    runs_precondition = 0.01 < bit_proportion < 0.99
    if runs_precondition:
        expected_runs = 2 * one_bits * (total_bits - one_bits) / total_bits + 1
        variance = (
            2 * one_bits * (total_bits - one_bits)
            * (2 * one_bits * (total_bits - one_bits) - total_bits)
            / (total_bits**2 * (total_bits - 1))
        )
        runs_z = (runs - expected_runs) / sqrt(variance)
        runs_p = erfc(abs(runs_z) / sqrt(2))
    else:
        expected_runs = runs_z = runs_p = None

    identical_run_count = 0
    longest_identical_run = 1
    current_run = 1
    for index in range(1, size):
        if data[index] == data[index - 1]:
            current_run += 1
            longest_identical_run = max(longest_identical_run, current_run)
        else:
            if current_run >= 4:
                identical_run_count += 1
            current_run = 1
    if current_run >= 4:
        identical_run_count += 1

    block_count = size // 16
    blocks = (data[index:index + 16] for index in range(0, block_count * 16, 16))
    seen_blocks = set()
    duplicate_blocks = 0
    for block in blocks:
        if block in seen_blocks:
            duplicate_blocks += 1
        else:
            seen_blocks.add(block)

    mean = sum(data) / size
    variance_bytes = sum((value - mean) ** 2 for value in data) / size
    serial_correlations = {}
    for lag in (1, 2, 4, 16):
        lag_covariance = sum(
            (data[index] - mean) * (data[index + lag] - mean)
            for index in range(size - lag)
        ) / (size - lag)
        serial_correlations[f"lag_{lag}_byte_correlation"] = (
            lag_covariance / variance_bytes if variance_bytes else float("nan")
        )

    return {
        "bytes": size,
        "min_byte_count": min(byte_counts.values()),
        "max_byte_count": max(byte_counts.values()),
        "byte_chi_square": byte_chi_square,
        "byte_chi_square_pass_5pct": byte_chi_square < 293.25,
        "one_bit_fraction": bit_proportion,
        "monobit_p_value": monobit_p,
        "runs": runs,
        "runs_expected": expected_runs,
        "runs_p_value": runs_p,
        "runs_precondition_met": runs_precondition,
        "identical_byte_runs_of_at_least_4": identical_run_count,
        "longest_identical_byte_run": longest_identical_run,
        "duplicate_16_byte_blocks": duplicate_blocks,
        **serial_correlations,
    }

results = {}
samples = {}
for label, path in SAMPLES.items():
    samples[label] = path.read_bytes()
    results[label] = analyze(samples[label])
byte_frequencies = {label: Counter(data) for label, data in samples.items()}

for label, result in results.items():
    print(f"\n{label}")
    for name, value in result.items():
        print(f"  {name}: {value}")


Linux /dev/urandom
  bytes: 1048576
  min_byte_count: 3905
  max_byte_count: 4267
  byte_chi_square: 258.35009765625
  byte_chi_square_pass_5pct: True
  one_bit_fraction: 0.4999309778213501
  monobit_p_value: 0.6892897128886051
  runs: 4194478
  runs_expected: 4194304.920072317
  runs_p_value: 0.904865314384329
  runs_precondition_met: True
  identical_byte_runs_of_at_least_4: 0
  longest_identical_byte_run: 3
  duplicate_16_byte_blocks: 0
  lag_1_byte_correlation: -0.00042027638680118356
  lag_2_byte_correlation: -0.0027609578428958417
  lag_4_byte_correlation: -0.0005249461918107073
  lag_16_byte_correlation: 0.00027280009944387514

OpenSSL rand
  bytes: 1048576
  min_byte_count: 3893
  max_byte_count: 4276
  byte_chi_square: 281.5380859375
  byte_chi_square_pass_5pct: True
  one_bit_fraction: 0.5000796318054199
  monobit_p_value: 0.6446001070007051
  runs: 4193419
  runs_expected: 4194304.893611908
  runs_p_value: 0.5407100034571983
  runs_precondition_met: True
  identical_byte_ru

## Byte-frequency distribution

The table reports each sample's least- and most-frequent byte counts and the chi-square statistic. A value below the 5% critical value means the test did not reject uniform byte frequencies at that level; it does not mean the bytes are proven secure.

In [3]:
header = ("Source", "Bytes", "Min count", "Max count", "Chi-square", "< 293.25?")
print(f"{header[0]:28} {header[1]:>10} {header[2]:>10} {header[3]:>10} {header[4]:>12} {header[5]:>10}")
for label, result in results.items():
    print(
        f"{label:28} {result['bytes']:10d} {result['min_byte_count']:10d} "
        f"{result['max_byte_count']:10d} {result['byte_chi_square']:12.2f} "
        f"{str(result['byte_chi_square_pass_5pct']):>10}"
    )

Source                            Bytes  Min count  Max count   Chi-square  < 293.25?
Linux /dev/urandom              1048576       3905       4267       258.35       True
OpenSSL rand                    1048576       3893       4276       281.54       True
Windows .NET RNG                1048576       3836       4252       292.93       True


In [4]:
print(f"{'Byte':>6} {'Linux /dev/urandom':>20} {'OpenSSL rand':>16} {'Windows .NET RNG':>20}")
for byte_value in range(256):
    counts = [byte_frequencies[label][byte_value] for label in SAMPLES]
    print(f"0x{byte_value:02x} {counts[0]:20d} {counts[1]:16d} {counts[2]:20d}")

  Byte   Linux /dev/urandom     OpenSSL rand     Windows .NET RNG
0x00                 4173             4127                 4117
0x01                 4012             4137                 3988
0x02                 4065             3950                 4103
0x03                 4024             4140                 3991
0x04                 4008             4064                 4031
0x05                 4142             4140                 3959
0x06                 4079             4124                 4019
0x07                 4064             4108                 4129
0x08                 4087             4148                 4185
0x09                 4119             4058                 4073
0x0a                 4187             4134                 4166
0x0b                 4151             4038                 4109
0x0c                 4066             4131                 3956
0x0d                 4145             4066                 4090
0x0e                 4206             

## Interpretation and limitations

In the captured 1 MiB samples shown above, byte-frequency chi-square values were 258.35 (Linux), 281.54 (OpenSSL), and 292.93 (Windows .NET RNG), all below the 5% critical value of 293.25. The Windows result is close to the threshold; a correctly behaving source can occasionally fall on either side by chance. Monobit and runs p-values were above 0.01 for these captures, and none had repeated 16-byte blocks or identical-byte runs of four or more. These observations describe only these samples and do not rank or prove the security of the generators.

Treat p-values below 0.01 as a signal to investigate this sample and implementation, not as a verdict from one run. Repeatedly testing many samples also increases the chance of a false alarm. The runs test is not applicable when its bit-proportion precondition fails. The byte-correlation, repeated-run, and block-collision results are limited to the tested sample and lag/pattern size.

A cryptographic generator must resist prediction even when an attacker knows previous output. These tests do not test prediction resistance, seed quality, state compromise, reseeding, or operating-system implementation correctness. Use established operating-system cryptographic APIs rather than building a generator from observed output or statistical test results.

## OS mechanisms and comparison

**Linux:** the kernel random subsystem collects environmental noise to seed its cryptographic random-number generator. `/dev/urandom` provides output from this kernel generator and is the normal interface for cryptographic bytes once the system RNG is initialized. The interface does not return raw physical noise.

**OpenSSL:** `openssl rand` uses OpenSSL's library RNG/DRBG, which is seeded from operating-system entropy and maintains/reseeds cryptographic generator state. It is an application-library interface rather than a separate physical entropy source.

**Windows:** .NET `RandomNumberGenerator` is the high-level cryptographic API and delegates to a platform cryptographic provider; the implementation can depend on the .NET version. Windows CNG exposes system random bytes through its BCrypt interfaces and manages provider state. The API is preferred over general-purpose pseudo-random APIs such as `System.Random` for cryptographic use.

All three are intended for cryptographic use when accessed through their documented interfaces and used correctly. The tests can compare a few observable properties of these particular captures, but do not establish that one source is more secure. See the [lab README](README.md) for documentation references.